# 07 Silver Procedure Clean

## Purpose

This notebook creates the Silver Procedure table from the Bronze raw Procedure FHIR table.

## What We Are Doing

We will:
1. Read `healthcare_catalog.bronze.procedure_raw`
2. Extract procedure and intervention fields
3. Flatten procedure-related data
4. Standardize timestamps and references
5. Save the clean table into the Silver layer

## Why We Are Doing This

Procedure resources contain:
- surgeries
- interventions
- therapies
- imaging procedures
- clinical treatments

This dataset is important for:
- treatment analytics
- care pathway analysis
- utilization analytics
- cost analysis
- downstream ML features

## Expected Final Output

A clean Silver table:

`healthcare_catalog.silver.procedure_clean`

Expected columns:
- procedure_id
- patient_id
- encounter_id
- procedure_description
- procedure_status
- procedure_datetime

## Step 1 — Import PySpark Functions

### What We Are Doing

We are importing PySpark SQL functions.

### Why We Are Doing This

We need Spark functions to extract nested FHIR Procedure fields.

### Expected Output

Spark functions available for this notebook.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Bronze Procedure Table

### What We Are Doing

We are reading the Bronze Procedure table.

### Why We Are Doing This

The Bronze layer contains raw nested FHIR Procedure resources.

### Expected Output

A DataFrame named:

`procedure_raw_df`

In [0]:
procedure_raw_df = spark.table(
    "healthcare_catalog.bronze.procedure_raw"
)

print("Bronze procedure_raw table loaded successfully.")

Bronze procedure_raw table loaded successfully.


## Step 3 — Inspect Procedure Raw Schema

### What We Are Doing

We are printing the Procedure schema.

### Why We Are Doing This

FHIR Procedure resources are nested JSON structures.

We need to identify:
- patient references
- encounter references
- procedure descriptions
- status fields
- timestamps

### Expected Output

You should see fields such as:
- resource.id
- resource.subject.reference
- resource.encounter.reference
- resource.code
- resource.status
- resource.performedDateTime

In [0]:
procedure_raw_df.printSchema()

root
 |-- fullUrl: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- abatementDateTime: string (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- activity: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- detail: struct (nullable = true)
 |    |    |    |    |-- code: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |    |-- text: string (nullable = true)
 |    |    |    |    |-- location: struct (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- statu

## Step 4 — Extract Clean Procedure Columns

### What We Are Doing

We are extracting procedure and treatment-related fields from nested FHIR Procedure resources.

### Why We Are Doing This

Procedure resources contain:
- surgeries
- treatments
- therapies
- interventions
- imaging procedures

Healthcare analytics and ML models require flattened procedure tables.

### Fields We Will Extract

- procedure_id
- patient_reference
- encounter_reference
- procedure_description
- procedure_status
- procedure_start
- procedure_end

### Expected Output

A flattened DataFrame named:

`procedure_clean_df`

In [0]:
procedure_clean_df = procedure_raw_df.select(

    col("resource.id").alias("procedure_id"),

    col("resource.subject.reference").alias("patient_reference"),

    col("resource.encounter.reference").alias("encounter_reference"),

    get_json_object(col("resource.code"), "$.text").alias("procedure_description"),

    col("resource.status").alias("procedure_status"),

    col("resource.performedPeriod.start").alias("procedure_start"),

    col("resource.performedPeriod.end").alias("procedure_end")
)

print("Procedure clean DataFrame created successfully.")

Procedure clean DataFrame created successfully.


## Step 5 — Convert Procedure Timestamps

### What We Are Doing

We are converting procedure timestamps into Spark timestamp format.

### Why We Are Doing This

Healthcare treatment analytics require proper timestamp columns.

This supports:
- care pathway analytics
- treatment timelines
- procedure utilization analytics

### Expected Output

Procedure timestamps converted successfully.

In [0]:
procedure_clean_df = procedure_clean_df.withColumn(
    "procedure_start",
    to_timestamp(col("procedure_start"))
)

procedure_clean_df = procedure_clean_df.withColumn(
    "procedure_end",
    to_timestamp(col("procedure_end"))
)

print("Procedure timestamps converted successfully.")

Procedure timestamps converted successfully.


## Step 6 — Extract Clean Patient and Encounter IDs

### What We Are Doing

We are extracting clean UUIDs from FHIR references.

### Why We Are Doing This

FHIR references contain:
- urn:uuid:
- ResourceType/ID

We need normalized IDs for joins across Silver tables.

### Expected Output

New columns:
- patient_id
- encounter_id

In [0]:
procedure_clean_df = procedure_clean_df.withColumn(
    "patient_id",

    regexp_extract(
        col("patient_reference"),
        r'urn:uuid:(.*)',
        1
    )
)

procedure_clean_df = procedure_clean_df.withColumn(
    "encounter_id",

    regexp_extract(
        col("encounter_reference"),
        r'urn:uuid:(.*)',
        1
    )
)

print("Patient and encounter IDs extracted successfully.")

Patient and encounter IDs extracted successfully.


## Step 7 — Calculate Procedure Duration

### What We Are Doing

We are calculating procedure duration in hours.

### Why We Are Doing This

Procedure duration is important for:
- operational analytics
- utilization analytics
- treatment complexity analysis

### Expected Output

A new column:

`procedure_duration_hours`

In [0]:
procedure_clean_df = procedure_clean_df.withColumn(

    "procedure_duration_hours",

    (
        unix_timestamp(col("procedure_end")) -
        unix_timestamp(col("procedure_start"))
    ) / 3600

)

display(procedure_clean_df)

procedure_id,patient_reference,encounter_reference,procedure_description,procedure_status,procedure_start,procedure_end,patient_id,encounter_id,procedure_duration_hours
ef7682b2-8c7f-bcd5-72f3-8bbfed8b64bd,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:80454cdb-2a5e-cc3a-f15f-2e6c378e339a,Removal of intrauterine device,completed,2013-01-20T06:57:45.000Z,2013-01-20T07:58:15.000Z,21dde8d5-5497-8bf3-aa32-b79e998e9024,80454cdb-2a5e-cc3a-f15f-2e6c378e339a,1.0083333333333333
2d00ae9d-e663-99d6-a120-305959219868,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:e5186888-3794-a79b-4c45-76dd877dba8b,Standard pregnancy test,completed,2013-03-17T06:57:45.000Z,2013-03-17T07:12:45.000Z,21dde8d5-5497-8bf3-aa32-b79e998e9024,e5186888-3794-a79b-4c45-76dd877dba8b,0.25
a0573875-88f5-e089-2dec-d1ad3d1d3439,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:e5186888-3794-a79b-4c45-76dd877dba8b,Ultrasound scan for fetal viability,completed,2013-03-17T06:57:45.000Z,2013-03-17T07:12:45.000Z,21dde8d5-5497-8bf3-aa32-b79e998e9024,e5186888-3794-a79b-4c45-76dd877dba8b,0.25
7f9bc370-5e07-91d5-25c9-f3f8b891d81f,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:e5186888-3794-a79b-4c45-76dd877dba8b,Evaluation of uterine fundal height,completed,2013-03-17T06:57:45.000Z,2013-03-17T07:12:45.000Z,21dde8d5-5497-8bf3-aa32-b79e998e9024,e5186888-3794-a79b-4c45-76dd877dba8b,0.25
85004c92-4d8e-78df-3fce-1798aeaccef2,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:e5186888-3794-a79b-4c45-76dd877dba8b,Auscultation of the fetal heart,completed,2013-03-17T06:57:45.000Z,2013-03-17T07:12:45.000Z,21dde8d5-5497-8bf3-aa32-b79e998e9024,e5186888-3794-a79b-4c45-76dd877dba8b,0.25
fa62af07-eaf4-3b04-bef3-6386d4fe024a,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:e5186888-3794-a79b-4c45-76dd877dba8b,"Blood typing, RH typing",completed,2013-03-17T06:57:45.000Z,2013-03-17T07:12:45.000Z,21dde8d5-5497-8bf3-aa32-b79e998e9024,e5186888-3794-a79b-4c45-76dd877dba8b,0.25
a8fa175a-a361-1cf8-4a3b-eda7d9563e44,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:e5186888-3794-a79b-4c45-76dd877dba8b,Hemoglobin / Hematocrit / Platelet count,completed,2013-03-17T06:57:45.000Z,2013-03-17T07:12:45.000Z,21dde8d5-5497-8bf3-aa32-b79e998e9024,e5186888-3794-a79b-4c45-76dd877dba8b,0.25
1a8bf5e2-2e0b-7632-3861-b462225f6d86,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:e5186888-3794-a79b-4c45-76dd877dba8b,Hepatitis B Surface Antigen Measurement,completed,2013-03-17T06:57:45.000Z,2013-03-17T07:12:45.000Z,21dde8d5-5497-8bf3-aa32-b79e998e9024,e5186888-3794-a79b-4c45-76dd877dba8b,0.25
2bd92477-ce80-7687-9594-1d7144ae117f,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:e5186888-3794-a79b-4c45-76dd877dba8b,Human immunodeficiency virus antigen test,completed,2013-03-17T06:57:45.000Z,2013-03-17T07:12:45.000Z,21dde8d5-5497-8bf3-aa32-b79e998e9024,e5186888-3794-a79b-4c45-76dd877dba8b,0.25
6dcb336d-f101-2215-2f8e-0dfa8c3b7af3,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:e5186888-3794-a79b-4c45-76dd877dba8b,Chlamydia antigen test,completed,2013-03-17T06:57:45.000Z,2013-03-17T07:12:45.000Z,21dde8d5-5497-8bf3-aa32-b79e998e9024,e5186888-3794-a79b-4c45-76dd877dba8b,0.25


## Step 8 — Check Most Common Procedures

### What We Are Doing

We are counting the most common clinical procedures.

### Why We Are Doing This

This helps us understand:
- treatment patterns
- utilization
- clinical intervention types

### Expected Output

Top procedure frequency table.

In [0]:
display(

    procedure_clean_df.groupBy(
        "procedure_description"
    ).count().orderBy(
        desc("count")
    )

)

procedure_description,count
Assessment of health and social care needs (procedure),3840
Depression screening (procedure),3391
Depression screening using Patient Health Questionnaire Two-Item score (procedure),2985
Assessment of substance use (procedure),2761
Medication Reconciliation (procedure),2645
Assessment of anxiety (procedure),2194
Auscultation of the fetal heart,1316
Evaluation of uterine fundal height,1316
Screening for drug abuse (procedure),1208
Assessment using Alcohol Use Disorders Identification Test - Consumption (procedure),1202


## Step 9 — Check Null Values

### What We Are Doing

We are validating missing procedure fields.

### Why We Are Doing This

Clinical treatment data can contain incomplete timestamps or references.

Silver validation is critical before downstream analytics.

### Expected Output

Null count summary.

In [0]:
display(

    procedure_clean_df.select(

        [
            sum(col(column_name).isNull().cast("int")).alias(column_name)

            for column_name in procedure_clean_df.columns
        ]

    )

)

procedure_id,patient_reference,encounter_reference,procedure_description,procedure_status,procedure_start,procedure_end,patient_id,encounter_id,procedure_duration_hours
0,0,0,0,0,0,0,0,0,0


## Step 10 — Save Silver Procedure Table

### What We Are Doing

We are saving the clean procedure table into the Silver layer.

### Why We Are Doing This

The Silver layer stores:
- cleaned
- normalized
- analytics-ready

procedure and intervention data.

### Expected Output

A Delta table:

`healthcare_catalog.silver.procedure_clean`

In [0]:
procedure_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.silver.procedure_clean")

print("Silver procedure_clean table saved successfully.")

Silver procedure_clean table saved successfully.


In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.silver
""").show(truncate=False)

+--------+-----------------+-----------+
|database|tableName        |isTemporary|
+--------+-----------------+-----------+
|silver  |condition_clean  |false      |
|silver  |encounter_clean  |false      |
|silver  |observation_clean|false      |
|silver  |patient_clean    |false      |
|silver  |procedure_clean  |false      |
+--------+-----------------+-----------+

